In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install pandas numpy pymannkendall pyarrow matplotlib

In [ ]:
# !/usr/bin/env python3
"""
paper3_equity_metrics.py
========================
Compute Paper 3's headline equity metrics from the merged table that has BOTH
the re-harvested authorship country vectors AND the extracted study location.

Definitions (following Rees 2021 / Kilmarx 2022):
  target_iso (per paper) = its study_countries (the place it is ABOUT).
  local_first  = the 'first' author sits in a study country.
  local_last   = the 'last' (senior) author sits in a study country.
  parachute    = NO author on the paper sits in any study country.
                 (set to NA when the paper has no resolved author country at all,
                  so missing-affiliation records don't masquerade as parachutes.)

Cohorts are built by bloc: a paper is "about SSA" if any study country is in SSA,
"about Maghreb" likewise. SSA and Maghreb are reported SEPARATELY (your design).
The same functions run on a comparator corpus (trypanosomiasis) and on the global
tick baseline — pass the matching mask and a label.

Outputs per cohort: n_papers, local_first_share, local_last_share, parachute_rate,
first/last leadership Gini, and a Mann-Kendall + Theil-Sen trend on each annual
share series.

DEPENDENCIES
    pip install pandas numpy pymannkendall pyarrow
    # matplotlib only if you want the figures
"""

import numpy as np
import pandas as pd

MIN_PAPERS_PER_YEAR = 5     # drop noisy early years before trend testing
MK_MIN_POINTS = 8           # Mann-Kendall needs >= ~8-10 yearly points


# --------------------------------------------------------------------------- #
# helpers                                                                     #
# --------------------------------------------------------------------------- #
def to_set(s, sep=";"):
    if s is None:
        return set()
    return {x for x in str(s).split(sep) if x and x.lower() != "nan"}


def paper_flags(row) -> dict | None:
    """Locality flags for one paper; None if study location is undefined."""
    target = to_set(row.get("study_countries"))
    if not target:
        return None  # no aboutness -> cannot define locality
    first = to_set(row.get("first_author_countries"))
    last = to_set(row.get("last_author_countries"))
    allc = to_set(str(row.get("all_author_countries", "")).replace("|", ";"))

    parachute = np.nan if not allc else (len(allc & target) == 0)
    blocs = to_set(row.get("study_bloc"))
    return {
        "year": pd.to_numeric(row.get("oa_year"), errors="coerce"),
        "local_first": bool(first & target),
        "local_last": bool(last & target),
        "parachute": parachute,
        "about_ssa": "ssa" in blocs,
        "about_maghreb": "maghreb" in blocs,
        "has_study_country": True,
        "first_countries": row.get("first_author_countries", ""),
        "last_countries": row.get("last_author_countries", ""),
    }


def build_frame(df: pd.DataFrame) -> pd.DataFrame:
    """One row per paper that has a resolvable study country."""
    recs = [f for f in (paper_flags(r) for _, r in df.iterrows()) if f]
    frame = pd.DataFrame(recs)
    frame = frame.dropna(subset=["year"])
    frame["year"] = frame["year"].astype(int)
    print(f"[frame] {len(frame):,} papers with a study country "
          f"(of {len(df):,} total)")
    return frame


def gini(values) -> float:
    x = np.sort(np.asarray(list(values), dtype=float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return float("nan")
    cum = np.cumsum(x)
    return float((n + 1 - 2 * (cum / x.sum()).sum()) / n)


def leadership_gini(sub: pd.DataFrame, countries_col: str):
    """Gini over the distribution of (fractional) authorships across countries.
    High = leadership of research about this bloc is concentrated in few countries."""
    from collections import Counter
    c = Counter()
    for s in sub[countries_col]:
        cs = to_set(s)
        if not cs:
            continue
        w = 1.0 / len(cs)          # fractional: split a multi-country author
        for x in cs:
            c[x] += w
    return gini(c.values()), dict(c)


def annual_share(sub: pd.DataFrame, col: str) -> pd.Series:
    """Yearly mean of a 0/1 (or 0/1/NA) column, keeping only well-populated years."""
    g = sub.dropna(subset=[col])
    counts = g.groupby("year")[col].size()
    means = g.groupby("year")[col].mean()
    keep = counts[counts >= MIN_PAPERS_PER_YEAR].index
    return means.loc[means.index.isin(keep)].sort_index()


def mk_trend(series: pd.Series) -> dict:
    import pymannkendall as mk
    s = series.dropna()
    if len(s) < MK_MIN_POINTS:
        return {"trend": "insufficient_data", "p": None, "slope": None, "n": int(len(s))}
    r = mk.original_test(s.values)
    return {"trend": r.trend, "p": round(float(r.p), 4),
            "slope_per_yr": round(float(r.slope), 5), "n": int(len(s))}


# --------------------------------------------------------------------------- #
# cohort metric bundle                                                        #
# --------------------------------------------------------------------------- #
def cohort_metrics(frame: pd.DataFrame, mask: pd.Series, label: str) -> dict:
    sub = frame[mask]
    par = sub["parachute"].dropna()
    fg, _ = leadership_gini(sub, "first_countries")
    lg, _ = leadership_gini(sub, "last_countries")

    out = {
        "cohort": label,
        "n_papers": int(len(sub)),
        "local_first_share": round(sub["local_first"].mean(), 3),
        "local_last_share": round(sub["local_last"].mean(), 3),
        "parachute_rate": round(par.mean(), 3) if len(par) else float("nan"),
        "n_parachute_evaluable": int(len(par)),
        "n_no_author_country": int(sub["parachute"].isna().sum()),
        "first_author_gini": round(fg, 3),
        "last_author_gini": round(lg, 3),
        "trend_local_first": mk_trend(annual_share(sub, "local_first")),
        "trend_local_last": mk_trend(annual_share(sub, "local_last")),
        "trend_parachute": mk_trend(annual_share(sub, "parachute")),
    }
    return out


def comparison_table(frame: pd.DataFrame, extra_cohorts: dict | None = None
                     ) -> pd.DataFrame:
    """
    Headline table: SSA vs Maghreb vs global-tick baseline.
    extra_cohorts: {label: boolean mask} — e.g. a trypanosomiasis frame's masks,
    or single-bloc-strict variants. Pass comparator corpora processed through the
    SAME pipeline as additional rows for the benchmark.
    """
    rows = [
        cohort_metrics(frame, frame["about_ssa"], "tick_about_SSA"),
        cohort_metrics(frame, frame["about_maghreb"], "tick_about_Maghreb"),
        cohort_metrics(frame, pd.Series(True, index=frame.index), "tick_global_baseline"),
    ]
    if extra_cohorts:
        for label, mask in extra_cohorts.items():
            rows.append(cohort_metrics(frame, mask, label))
    # flatten the nested trend dicts for a readable table
    flat = []
    for r in rows:
        row = {k: v for k, v in r.items() if not isinstance(v, dict)}
        for tkey in ("trend_local_first", "trend_local_last", "trend_parachute"):
            t = r[tkey]
            row[f"{tkey}_dir"] = t["trend"]
            row[f"{tkey}_p"] = t["p"]
            row[f"{tkey}_slope"] = t.get("slope_per_yr")
        flat.append(row)
    return pd.DataFrame(flat)


# --------------------------------------------------------------------------- #
# optional figures                                                            #
# --------------------------------------------------------------------------- #
def plot_annual(frame, out_png="paper3_trends.png"):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, (label, mask) in zip(axes, [
        ("about SSA", frame["about_ssa"]),
        ("about Maghreb", frame["about_maghreb"]),
        ("global tick", pd.Series(True, index=frame.index))]):
        sub = frame[mask]
        for col, name in [("local_first", "local first"),
                          ("local_last", "local last (senior)"),
                          ("parachute", "parachute")]:
            s = annual_share(sub, col)
            ax.plot(s.index, s.values, marker="o", ms=3, label=name)
        ax.set_title(f"{label}  (n={len(sub):,})"); ax.set_xlabel("year")
        ax.axhline(0.5, ls=":", c="grey", lw=0.8)
    axes[0].set_ylabel("share of papers"); axes[0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out_png, dpi=150)
    print(f"[plot] wrote {out_png}")


def lorenz_points(frame, mask, countries_col="last_countries"):
    """Return (x, y) for a Lorenz curve of leadership concentration, to plot."""
    _, counts = leadership_gini(frame[mask], countries_col)
    x = np.sort(np.asarray(list(counts.values()), dtype=float))
    if x.sum() == 0:
        return [0, 1], [0, 1]
    cum = np.cumsum(x) / x.sum()
    return np.insert(np.arange(1, len(x)+1)/len(x), 0, 0), np.insert(cum, 0, 0)


# # --------------------------------------------------------------------------- #
# if __name__ == "__main__":
#     IN = "/content/drive/MyDrive/.../with_study_location.parquet"
#     df = pd.read_parquet(IN)
#     frame = build_frame(df)
#     table = comparison_table(frame)
#     pd.set_option("display.max_columns", None, "display.width", 200)
#     print(table.to_string(index=False))
#     table.to_csv("paper3_equity_table.csv", index=False)
#     # plot_annual(frame)   # uncomment for the trend figure

In [ ]:
# if __name__ == "__main__":
IN = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_study_location.parquet"
df = pd.read_parquet(IN)
frame = build_frame(df)
table = comparison_table(frame)
pd.set_option("display.max_columns", None, "display.width", 200)
print(table.to_string(index=False))
table.to_csv("paper3_equity_table.csv", index=False)
plot_annual(frame)   # uncomment for the trend figure